# SQL Básico com PySpark

## Antes de começar: tipos de dados e *schema*

Quando criamos uma tabela (seja em um banco relacional tradicional ou no Spark), precisamos dizer, para cada coluna, que tipo de dado ela vai guardar. Isso é importante porque o tipo define:

- quanto espaço em memória/disco a coluna ocupa;
- quais operações fazem sentido nela (não dá pra fazer `AVG()` de uma coluna de texto, por exemplo);
- como os valores são comparados e ordenados.

O conjunto de colunas de uma tabela, com seus respectivos nomes e tipos, é chamado de schema (esquema). É basicamente a "planta baixa" da tabela. No Spark, podemos sempre visualizar o schema de um DataFrame com `df.printSchema()`.

**Principais tipos de dados que vamos usar (e que já vêm importados na célula abaixo):**

| Tipo Spark | Equivalente em SQL | Uso |
|---|---|---|
| `StringType` | `VARCHAR` / `TEXT` | Texto (nomes, cidades, categorias) |
| `IntegerType` / `LongType` | `INT` / `BIGINT` | Números inteiros |
| `FloatType` / `DoubleType` | `FLOAT` / `DOUBLE` | Números decimais (atenção: podem ter pequenos erros de arredondamento) |
| `DecimalType` | `DECIMAL(p,s)` | Números decimais exatos — ideal para dinheiro |
| `BooleanType` | `BOOLEAN` | Verdadeiro/falso |
| `DateType` / `TimestampType` | `DATE` / `TIMESTAMP` | Datas e datas com hora |
| `ArrayType` / `MapType` | — | Coleções (listas e dicionários dentro de uma célula) |

Na aula de hoje vamos usar principalmente `INT`, `VARCHAR` e `FLOAT`, mas é bom já conhecer os outros nomes, porque vão aparecer quando vocês lerem documentação ou mensagens de erro do Spark.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, FloatType, DoubleType,
    BooleanType, DateType, TimestampType, BinaryType, ArrayType, MapType, DecimalType
)
import datetime
import decimal
from pyspark.sql.functions import col
from pyspark.sql.functions import lit

spark = SparkSession.builder.getOrCreate()

**CRIAR TABELA**

In [ ]:
spark.sql("""
    CREATE TABLE teste (
        id INT,
        nome VARCHAR(30),
        idade INT
    )
""")

**ALTERAÇÕES NA TABELA**


*   Renomear
*   Alterar tipo de dado
*   Adicionar nova coluna
*   Deletar

In [ ]:
df = spark.table("nome_tabela")
df = df.withColumnRenamed("coluna_antiga", "coluna_nova")

In [ ]:
df = df.withColumn("coluna", col("coluna").cast("tipo"))

In [ ]:
df.printSchema()

In [ ]:
df = df.withColumn("nova_coluna", lit("valor"))

In [ ]:
spark.sql("DROP TABLE nome_tabela")

**MEXER COM DADOS**

In [ ]:
# CREATE TABLE
spark.sql("""
    CREATE TABLE estatisticos (
        id INT,
        nome VARCHAR(30),
        profissao VARCHAR(30),
        cidade VARCHAR(30),
        salario FLOAT,
        bonus FLOAT,
        nota FLOAT

    )
""")

In [ ]:
# INSERT INTO
spark.sql("""
    INSERT INTO estatisticos VALUES
      (1, 'Neyman', 'jornalista', 'sao carlos', 4000, 250, 1.14),
      (2, 'Bayes', 'funileiro', 'sao paulo', 4000, 500, 3.9),
      (3, 'William Gosset', 'padeiro', 'sao carlos', 3000, 0, 4.0),
      (4, 'Mahalanobis', 'policial', 'sao paulo', 3000, 0, 3.5),
      (5, 'Forsythe', 'professor', 'sao carlos', 2500, 300, 4.1),
      (6, 'Poisson', 'pintor', 'sao carlos', 2000, 250, 2.9),
      (7, 'Cauchy', 'padeiro', 'sao paulo', 8000, 200, 4.9),
      (8, 'Viola', 'professor', 'sao carlos', 10000, 500, 5.0),
      (9, 'Kolmogorov', 'escritor', 'sao paulo', 7000, 7000, 4.8),
      (10, 'Pearson', 'padeiro', 'sao carlos', 20000, 150, 4.35),
      (11, 'Gauss', 'professor', 'sao paulo', 1750, 500, 3.75),
      (12, 'Fisher', 'pescador', 'sao carlos', 1700, 0, 2.7),
      (13, 'Chebychev', 'carteiro', 'sao paulo', 1800, 300, 4.0),
      (14, 'Markov', 'apostador', 'sao paulo', 9000, 17500, 0.0),
      (15, 'Blackwell', 'porteiro', 'sao carlos', 2000, 500, 3.0),
      (16, 'Liliefors', 'agricultor', 'sao carlos', 10000, 0, 4.5),
      (17, 'Shapiro', 'professor', 'sao carlos', 8000, 400, 3.5),
      (18, 'Kruskal', 'vidente', 'sao paulo', 30000, 0, 0.1),
      (19, 'Kendall', 'adestrador', 'sao carlos', 2500, 0, 4.4),
      (20, 'Friedman', 'jogador', 'sao paulo', 50000, 2500, 2.1),
      (21, 'Wilcoxon', 'caminhoneiro', 'sao carlos', 3000, 700, 3.1)
""")

**CONSULTAS EM SQL**

In [ ]:
# SELECT
spark.sql("SELECT * FROM nome_tabela").show()

In [ ]:
# SELECT
spark.sql("SELECT coluna1, coluna2 FROM nome_tabela").show()

In [ ]:
# ORDER BY - DESC ou ASC
spark.sql("""SELECT coluna1, coluna2
FROM nome_tabela
ORDER BY coluna2 DESC
""").show()

In [ ]:
# MAX() e MIN()
spark.sql("""SELECT MAX(coluna) AS maior_valor
FROM nome_tabela
""").show()

spark.sql("""SELECT MIN(coluna) AS menor_valor
FROM nome_tabela
""").show()

In [ ]:
# LIMIT
spark.sql("""SELECT coluna1, coluna2
FROM nome_tabela
ORDER BY coluna2 DESC
LIMIT n
""").show()

In [ ]:
# GROUP BY
spark.sql("""SELECT coluna_grupo, MAX(coluna_numerica)
FROM nome_tabela
GROUP BY coluna_grupo
""").show()

In [ ]:
# operações entre colunas e AS
spark.sql("""SELECT coluna1 + coluna2 AS resultado, coluna1, coluna2
FROM nome_tabela
""").show()

# Também podem ser usadas -, * e / no lugar de +.

**Operações calculadas**


In [ ]:
# ROUND e AVG
spark.sql("""SELECT coluna_grupo, ROUND(AVG(coluna_numerica), 2) AS media
FROM nome_tabela
GROUP BY coluna_grupo
ORDER BY media DESC
LIMIT n
""").show()

In [ ]:
# COUNT
spark.sql("SELECT coluna_grupo, COUNT(coluna_grupo) FROM nome_tabela GROUP BY coluna_grupo").show()

In [ ]:
# WHERE
spark.sql("""SELECT coluna1, coluna2
FROM nome_tabela
WHERE coluna2 > valor
""").show()

In [ ]:
# WHERE + GROUP BY
spark.sql("""SELECT ROUND(AVG(coluna_numerica), 2) AS media, coluna_grupo
FROM nome_tabela
WHERE coluna_grupo = 'categoria'
GROUP BY coluna_grupo
""").show()